# MLIP advanced simulations: NPT molecular dynamics with a Monte Carlo Barostat

This notebook builds on the [simulation tutorial](simulation_tutorial.ipynb) and demonstrates how to run **NPT (constant pressure–temperature) molecular dynamics** using the Monte Carlo Barostat implemented in the *mlip* library.

In the NVT simulations shown in the introductory tutorial, the simulation box is fixed and only the temperature is controlled. NPT simulations additionally allow the **volume to fluctuate** so that the system relaxes to the correct density at a given target pressure. This is important for:
- Correctly equilibrating liquid or solid systems at ambient conditions,
- Studying pressure-dependent properties,
- Avoiding artefacts from an incorrectly sized simulation box.

### Why NPT matters in practice

**Biomolecular simulations.** Most experimental data on proteins, nucleic acids and small molecules in solution is collected at atmospheric pressure (≈ 1 atm). Running NVT instead of NPT requires the box to be sized *exactly* to the correct equilibrium density beforehand — any error directly biases structural and thermodynamic observables such as solvation free energies, binding affinities and diffusion coefficients. NPT removes this sensitivity: the volume equilibrates automatically, giving the right solvent density around the solute without manual tuning. This is particularly important when changing temperature, switching force fields, or adding co-solvents such as ions, because each of these shifts the equilibrium density.

**Materials simulations.** NPT is equally important in materials science, where a fixed cell introduces artificial stress that can bias phase transitions, defect migration and elastic properties. Allowing the cell to relax is essential for computing accurate lattice parameters, thermal expansion coefficients and equations of state.

The *mlip* library implements pressure control via a **Monte Carlo Barostat**: at regular intervals during the MD trajectory, an isotropic volume change is proposed and accepted or rejected according to a Metropolis criterion. This is equivalent to the barostat used in OpenMM <sup>[1]</sup> and is fully JIT-compiled for GPU performance.

**This notebook showcases:**
- **How to load a pre-trained foundation model** suited for aqueous systems
- **How to prepare the molecular topology** required by the MC Barostat (`molecule_indices`)
- **How to configure and run an NPT simulation** of a solvated NaCl system
- **How to access the simulation results**, including the time-evolving simulation cell

---
<sup>[1]</sup> Eastman et al. *OpenMM 7: Rapid development of high performance algorithms for molecular dynamics.* PLOS Comput. Biol. 13(7): e1005659, 2017.

**Install and logging setup**

As in the introductory tutorial, we first install the *mlip* library and set up logging. We also download the input structure for this tutorial from InstaDeep's [HuggingFace collection](https://huggingface.co/collections/InstaDeepAI/ml-interatomic-potentials-68134208c01a954ede6dae42).

In [ ]:
%pip install "mlip[cuda]" huggingface_hub matplotlib

# Use this instead for installation without GPU:
# %pip install mlip huggingface_hub matplotlib

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, force=True, format="%(levelname)s - %(message)s"
)

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns="advanced_simulation/*",
    local_dir=".",
)

In [ ]:
import jax

print(jax.devices())

## 1. Loading a pre-trained MLIP model

For this tutorial we simulate a small box of liquid water with a dissolved NaCl ion pair. The system contains atoms of species H, O, Na and Cl.

We use the **ViSNet organics foundation model**, which was trained on a broad range of bio-chemical systems and covers the 15 atomic species: B, Br, C, Cl, F, H, I, K, Li, N, Na, O, P, S, Si — all four species in our system are included. The ViSNet, MACE and NequIP organics models (commented out below) are equally valid alternatives and cover the same set of species.

These foundation models were trained with the v2 codebase, so we load them using the `mlip.models` classes. If you would like to use a v1 model, you need to use the`mlip.models_v1` classes. Then, the `load_model_from_zip` function handles the v1→v2 parameter translation automatically.

In [ ]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="InstaDeepAI/mlip_models_organics_v2",
    filename="visnet_organics_02.zip",
    local_dir="pretrained_models/",
)

# hf_hub_download(
#     repo_id="InstaDeepAI/mlip_models_organics_v2",
#     filename="mace_organics_02.zip",
#     local_dir="pretrained_models/",
# )
# hf_hub_download(
#     repo_id="InstaDeepAI/mlip_models_organics_v2",
#     filename="nequip_organics_02.zip",
#     local_dir="pretrained_models/",
# )
# hf_hub_download(
#     repo_id="InstaDeepAI/mlip_models_organics_v2",
#     filename="esen_organics_02.zip",
#     local_dir="pretrained_models/",
# )

In [ ]:
from mlip.models import Visnet  # (1) we use ViSNet for this example.
from mlip.models.model_io import load_model_from_zip

organics_model_path = "pretrained_models/visnet_organics_02.zip"

# from mlip.models import Mace
# organics_model_path = "pretrained_models/mace_organics_02.zip"
# from mlip.models import Nequip
# organics_model_path = "pretrained_models/nequip_organics_02.zip"
# from mlip.models import Esen
# organics_model_path = "pretrained_models/esen_organics_02.zip"

force_field = load_model_from_zip(Visnet, organics_model_path)

# Derive a short label from the zip filename for use in plots
model_label = organics_model_path.rsplit("/", maxsplit=1)[-1].replace(".zip", "")

## 2. Loading and inspecting the NaCl water box

The input structure is a pre-equilibrated cubic box containing 106 water molecules and one NaCl ion pair (320 atoms total, box length ≈ 14.7 Å). The atoms are ordered as consecutive OHH triplets for each water molecule, followed by Na and Cl.

In [ ]:
from ase.io import read as ase_read

# Path to the pre-equilibrated NaCl water box structure.
# If running locally, update this to point to your local copy of nacl_equil.xyz.
nacl_file = "advanced_simulation/nacl_equil.xyz"

atoms = ase_read(nacl_file)

print(f"Number of atoms : {len(atoms)}")
print(f"Species         : {set(atoms.get_chemical_symbols())}")
print(f"Cell (Å)        : {atoms.cell.lengths().round(3)}")
print(f"PBC             : {atoms.pbc}")

## 3. Building molecule indices

The Monte Carlo Barostat scales the simulation box **molecule by molecule**: it rescales the centre-of-mass of each molecule while preserving its internal geometry (bond lengths and angles). This avoids unphysical bond stretching when the volume changes.

To enable this, we need to provide a `molecule_indices` list of length `num_atoms` that assigns each atom to its parent molecule. For example, for a system of three consecutive water molecules the list would be `[0, 0, 0, 1, 1, 1, 2, 2, 2]`.

Because atoms in this structure are ordered as consecutive OHH triplets followed by a single Na and a single Cl, we can build `molecule_indices` directly:

In [ ]:
import numpy as np

symbols = atoms.get_chemical_symbols()

num_water = symbols.count("O")  # 106
# Each water molecule occupies 3 consecutive atoms (O, H, H)
water_indices = np.repeat(np.arange(num_water), 3).tolist()
# Na and Cl are treated as single-atom molecules
ion_indices = [num_water, num_water + 1]

molecule_indices = water_indices + ion_indices

print(f"Total atoms         : {len(molecule_indices)}")
print(f"Number of molecules : {max(molecule_indices) + 1}")
print(f"First 9 entries     : {molecule_indices[:9]}  (three water molecules)")
print(f"Last  2 entries     : {molecule_indices[-2:]}  (Na, Cl)")

## 4. Configuring the NPT simulation engine

Switching from NVT to NPT requires two changes in the configuration compared to the introductory tutorial:

1. Set `md_integrator` to `MDIntegrator.NPT_MC_LANGEVIN` — this activates the Monte Carlo Barostat alongside the Langevin thermostat.
2. Provide `molecule_indices` — the per-atom molecule assignment built above.

Two additional optional parameters control the barostat:
- `pressure_bar`: target pressure in bar (default 1.01325 bar = 1 atm).
- `barostat_update_interval`: how many MD steps between volume-change attempts (default 25).

All other options (number of steps, episodes, temperature, timestep) are identical to the NVT case.

In [ ]:
from mlip.simulation.enums import MDIntegrator
from mlip.simulation.jax_md import JaxMDSimulationEngine

config = JaxMDSimulationEngine.Config(
    md_integrator=MDIntegrator.NPT_MC_LANGEVIN,
    num_steps=1000,
    num_episodes=10,
    snapshot_interval=10,
    temperature_kelvin=300.0,
    pressure_bar=1.01325,  # 1 atm
    barostat_update_interval=10,  # attempt a volume move every 10 MD steps
    molecule_indices=molecule_indices,
)

## 5. Running the NPT simulation

With the force field, structure and config in place, we initialise the engine and run the simulation. As with NVT Jax-MD simulations, when run on GPU there is an upfront JAX compilation cost that is quickly amortised over the course of the simulation.

In [ ]:
md_engine = JaxMDSimulationEngine(atoms, force_field, config)

In [ ]:
md_engine.run()

## 6. Accessing simulation results

The simulation state is accessible via `md_engine.state`. In NPT simulations, in addition to atomic positions and temperatures, the **simulation cell is logged at every snapshot** so we can track how the box volume evolved during the run.

In [ ]:
md_state = md_engine.state

# Positions: shape (num_steps, num_atoms, 3)
print(f"Positions shape : {md_state.positions.shape}")

# Cell: shape (num_steps, 3, 3) — the full 3x3 cell matrix at each step
print(f"Cell shape      : {md_state.cell.shape}")

# Temperature: shape (num_steps,)
print(f"Temperature shape: {md_state.temperature.shape}")

In [ ]:
import numpy as np

# Extract box lengths along the diagonal at each step (cubic box)
box_lengths = np.linalg.norm(np.array(md_state.cell), axis=2)
volumes = np.prod(box_lengths, axis=1)

mean_volume = volumes.mean()
std_volume = volumes.std()
mean_temperature = md_state.temperature.mean()

print(f"Initial volume  : {volumes[0]:.2f} Å³")
print(f"Final volume    : {volumes[-1]:.2f} Å³")
print(f"Mean volume     : {mean_volume:.2f} ± {std_volume:.2f} Å³")
print(f"Mean temperature: {mean_temperature:.1f} K")

## 7. Analysis

### 7.1 Volume and temperature time series

The two most direct diagnostics of an NPT run are the **volume** and **temperature** traces. A well-equilibrated simulation should show both quantities fluctuating around a stable mean with no systematic drift. The amplitude of the volume fluctuations is physically meaningful: it is related to the isothermal compressibility of the system.

In [ ]:
import matplotlib.pyplot as plt

AMU_PER_ANGSTROM_CUBED_TO_G_PER_CM_CUBED = 1.66054

# Total system mass in amu; constant throughout the simulation.
total_mass_amu = atoms.get_masses().sum()

# Density in g/cm³: 1 amu/Å³ = 1.66054 g/cm³.
density = total_mass_amu * AMU_PER_ANGSTROM_CUBED_TO_G_PER_CM_CUBED / volumes

steps = np.arange(len(volumes))

fig, (ax_vol, ax_dens, ax_temp) = plt.subplots(
    3,
    1,
    figsize=(8, 7),
    sharex=True,
)

volume_mean = volumes.mean()
density_mean = density.mean()
temperature_mean = md_state.temperature.mean()

ax_vol.plot(steps, volumes, linewidth=0.8, color="steelblue")
ax_vol.axhline(
    volume_mean,
    color="steelblue",
    linestyle="--",
    linewidth=1.2,
    label=f"mean = {volume_mean:.1f} Å³",
)
ax_vol.set_ylabel("Volume (Å³)")
ax_vol.legend(frameon=False)

ax_dens.plot(steps, density, linewidth=0.8, color="seagreen")
ax_dens.axhline(
    density_mean,
    color="seagreen",
    linestyle="--",
    linewidth=1.2,
    label=f"mean = {density_mean:.4f} g/cm³",
)
ax_dens.set_ylabel("Density (g/cm³)")
ax_dens.legend(frameon=False)

ax_temp.plot(steps, md_state.temperature, linewidth=0.8, color="coral")
ax_temp.axhline(
    temperature_mean,
    color="coral",
    linestyle="--",
    linewidth=1.2,
    label=f"mean = {temperature_mean:.1f} K",
)
ax_temp.set_ylabel("Temperature (K)")
ax_temp.set_xlabel("Simulation step")
ax_temp.legend(frameon=False)

fig.tight_layout()
plt.show()

### 7.2 Water O–O radial distribution function

The **radial distribution function** g(r) describes how the local density of atoms varies as a function of distance from a reference atom, relative to the bulk average. For liquid water the O–O g(r) is a well-known structural benchmark with two characteristic features:

- **First peak at ≈ 2.8 Å** — corresponds to the nearest-neighbour hydrogen-bonding shell. Each water molecule donates and accepts two hydrogen bonds, giving a coordination number of ≈ 4.
- **Second peak at ≈ 4.5 Å** — corresponds to the second solvation shell, i.e. water molecules connected to the first shell via hydrogen bonds.
- **Approaches g(r) = 1** at large r, indicating recovery of bulk (uncorrelated) density.

We compare against the experimental O–O g(r) of liquid water at 295.1 K (the closest available temperature to 300 K) measured by high-energy X-ray scattering by Skinner & Benmore (2013) <sup>[2]</sup>. Agreement with this reference is a standard validation for water models. The shaded band shows the experimental uncertainty.

Here we compute the O–O g(r) averaged over all trajectory frames. Because the box volume fluctuates in NPT, the normalization uses the instantaneous volume at each frame. Note that with a short trajectory (100 snapshots here) the curve will be noisy — longer runs of several nanoseconds are typically needed for a fully converged RDF.

---
<sup>[2]</sup> Skinner, L.B. & Benmore, C.J. et al. *Benchmark oxygen-oxygen pair-distribution function of ambient water from x-ray diffraction measurements with a wide Q-range.* J. Chem. Phys. 138, 074506 (2013).

In [ ]:
N_BINS = 150

o_indices = [
    index for index, symbol in enumerate(atoms.get_chemical_symbols()) if symbol == "O"
]
n_oxygen = len(o_indices)

# Positions of oxygen atoms across all frames: (n_frames, n_oxygen, 3).
oxygen_positions = np.array(md_state.positions)[:, o_indices, :]
n_frames = len(oxygen_positions)

# Bin edges. r_max is half the mean box length, which is the minimum image
# convention limit.
mean_box_length = np.cbrt(volumes.mean())
r_max = mean_box_length / 2
r_edges = np.linspace(0, r_max, N_BINS + 1)
r_centres = 0.5 * (r_edges[:-1] + r_edges[1:])
shell_volumes = (4 / 3) * np.pi * (r_edges[1:] ** 3 - r_edges[:-1] ** 3)

# Accumulate the weighted histogram over frames. Weight by instantaneous volume
# so that the normalization accounts for box fluctuations.
rdf_accumulator = np.zeros(N_BINS)
upper_triangle_indices = np.triu_indices(n_oxygen, k=1)

for frame_idx in range(n_frames):
    positions = oxygen_positions[frame_idx]
    box_length = np.cbrt(volumes[frame_idx])

    pairwise_vectors = positions[:, None, :] - positions[None, :, :]
    pairwise_vectors -= box_length * np.round(pairwise_vectors / box_length)
    pairwise_distances = np.linalg.norm(pairwise_vectors, axis=-1)

    hist, _ = np.histogram(
        pairwise_distances[upper_triangle_indices],
        bins=r_edges,
    )
    rdf_accumulator += hist * volumes[frame_idx]

# g(r) = 2 V <n(r)> / (N² Δv_shell). The factor of 2 accounts for counting only
# the upper triangle.
rdf = rdf_accumulator * 2 / (n_frames * n_oxygen**2 * shell_volumes)

# Experimental O-O g(r) at 295.1 K, the closest available value to 300 K.
# Source: Skinner & Benmore, J. Chem. Phys. 138, 074506 (2013).
exp_data = np.loadtxt(
    "advanced_simulation/experimental_rdf_oo_water_skinner2013.csv",
    delimiter=",",
    skiprows=1,
)
experimental_r = exp_data[:, 0]
experimental_g = exp_data[:, 1]
experimental_error = exp_data[:, 2]

experimental_mask = experimental_r <= r_max

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    r_centres,
    rdf,
    linewidth=1.5,
    color="steelblue",
    label=f"{model_label} (this simulation)",
)
ax.fill_between(
    experimental_r[experimental_mask],
    experimental_g[experimental_mask] - experimental_error[experimental_mask],
    experimental_g[experimental_mask] + experimental_error[experimental_mask],
    color="tomato",
    alpha=0.25,
)
ax.plot(
    experimental_r[experimental_mask],
    experimental_g[experimental_mask],
    linewidth=1.5,
    color="tomato",
    linestyle="--",
    label="Experiment 295 K (Skinner & Benmore 2013)",
)
ax.axhline(1.0, color="gray", linestyle=":", linewidth=0.8)

ax.set_xlabel("r (Å)")
ax.set_ylabel("g(r)")
ax.set_title("Water O-O radial distribution function")
ax.set_xlim(0, r_max)
ax.set_ylim(bottom=0)
ax.legend(frameon=False)

fig.tight_layout()
plt.show()